In [13]:
# ===============================
# 1. Install and import packages
# ===============================
!pip install gradio --quiet

import pandas as pd
import numpy as np
import re
import joblib
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [14]:
# ===============================
# 2. Load dataset
# ===============================
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/spam_Emails_data.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:

# ===============================
# 3. Data cleaning
# ===============================
df = df.dropna(subset=['text'])
# Convert text to string
df['text'] = df['text'].astype(str)

# Basic text preprocessing
def clean_text(text):
    text = text.lower()                          # lowercase
    text = re.sub(r'\n', ' ', text)             # remove line breaks
    text = re.sub(r'\s+', ' ', text)            # remove extra spaces
    text = re.sub(r'http\S+', '', text)         # remove URLs
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)  # remove special chars
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [4]:
# Convert text to string
df['text'] = df['text'].astype(str)


In [16]:
# Encode labels
df['label'] = df['label'].map({'Ham': 0, 'Spam': 1})

In [17]:
# ===============================
# 4. Train-test split
# ===============================
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [19]:
# ===============================
# 5. Build pipeline: TF-IDF + Logistic Regression
# ===============================
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        max_features=20000,     # increased features for large dataset
        ngram_range=(1,2)       # unigrams + bigrams
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # handle any class imbalance
        solver='liblinear'
    ))
])

In [20]:

# Train the model
model.fit(X_train, y_train)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    solver='liblinear'))])

In [21]:
# ===============================
# 6. Evaluate model
# ===============================
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9776889347433583
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     20432
           1       0.97      0.99      0.98     18338

    accuracy                           0.98     38770
   macro avg       0.98      0.98      0.98     38770
weighted avg       0.98      0.98      0.98     38770

Confusion Matrix:
 [[19835   597]
 [  268 18070]]


In [22]:
# ===============================
# 7. Save the model
# ===============================
joblib.dump(model, "spam_detector_lr.pkl")
print("Model saved as spam_detector_lr.pkl")


Model saved as spam_detector_lr.pkl


In [23]:
# ===============================
# 8. Prediction function
# ===============================
def predict_email(email_text):
    text_clean = clean_text(email_text)
    pred = model.predict([text_clean])[0]
    return "Spam 🚨" if pred == 1 else "Ham ✅"

# Quick test
print(predict_email("Congratulations! You won a free prize"))

Spam 🚨


In [24]:

# ===============================
# 9. Launch Gradio demo
# ===============================
demo = gr.Interface(
    fn=predict_email,
    inputs=gr.Textbox(lines=6, placeholder="Paste email text here..."),
    outputs="text",
    title="Email Spam Detection (Improved)",
    description="TF-IDF + Logistic Regression model for Spam vs Ham classification on 200k+ emails"
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1713135cf8aea1d44b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
